# Minimal config: deploy Oracle CDC and query-based pipelines

This notebook deploys two sets of Oracle pipelines from the **same minimal config**: a file that only lists the tables to ingest (`tables.csv`). Everything else comes from `default_values` and `override_config`:

| Setting | Where it comes from |
|---|---|
| Tables (`source_schema`, `source_table_name`) | `tables.csv` |
| Oracle service name, connection, catalogs, schemas, project name | `default_values` (fill missing columns) |
| Classic compute | `override_config` (forced for every row) |
| Target table names | derived from `source_table_name` |
| Schedule (hourly), staging (target catalog/schema) | connector defaults |

Steps:
1. Set the parameters below.
2. Create the target schemas (if missing).
3. Generate one bundle with `oracle_integrated` (CDC) and one with `oracle_query_based`.
4. Deploy both bundles and run each job once.

In [ ]:
%pip install pyyaml

## 1. Parameters

Change these for your workspace. `CLASSIC_COMPUTE = True` is needed when the Oracle database only accepts connections from classic compute; otherwise pipelines run on serverless (the default).

In [ ]:
CONNECTION_NAME = 'my_oracle_connection'   # Unity Catalog connection to Oracle
ORACLE_SERVICE_NAME = 'ORCLPDB1'           # Oracle service name (CDB$ROOT service name for multitenant)
CATALOG = 'main'                           # Catalog for target tables, event logs, and staging
CDC_SCHEMA = 'tapworks_oracle_cdc'         # Target schema for the integrated CDC pipelines
QBC_SCHEMA = 'tapworks_oracle_qbc'         # Target schema for the query-based pipelines
CLASSIC_COMPUTE = False                    # True if the database only accepts classic compute
DATABRICKS_PROFILE = None                  # CLI profile when running outside Databricks, e.g. 'dogfood'

TABLES_CSV = 'tables.csv'
OUTPUT_DIR = 'deployment'

In [ ]:
import json
import os
import subprocess
import sys

import pandas as pd
from databricks.sdk import WorkspaceClient

sys.path.append(os.path.abspath('../../../src'))
from tapworks.core.runner import run_pipeline_generation

w = WorkspaceClient(profile=DATABRICKS_PROFILE) if DATABRICKS_PROFILE else WorkspaceClient()
username = w.current_user.me().user_name

targets = {
    'dev': {
        'workspace_host': w.config.host,
        'root_path': f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}',
    }
}
print(f"Workspace: {w.config.host} as {username}")

## 2. Minimal config

Only the tables to ingest.

In [ ]:
tables = pd.read_csv(TABLES_CSV)
tables

## 3. Defaults and overrides

- `default_values` fill columns that are missing or empty in the config. The same Oracle source and catalog are shared by both bundles; the project name and schemas differ per bundle.
- `override_config` forces values for every row, regardless of the config. Here it sets the compute for all pipelines.

In [ ]:
shared_defaults = {
    'source_database': ORACLE_SERVICE_NAME,
    'connection_name': CONNECTION_NAME,
    'target_catalog': CATALOG,
    'pipeline_catalog': CATALOG,
}

cdc_defaults = {**shared_defaults, 'project_name': 'tapworks_oracle_cdc', 'target_schema': CDC_SCHEMA, 'pipeline_schema': CDC_SCHEMA}
qbc_defaults = {**shared_defaults, 'project_name': 'tapworks_oracle_qbc', 'target_schema': QBC_SCHEMA, 'pipeline_schema': QBC_SCHEMA}

overrides = {'classic_compute': 'true' if CLASSIC_COMPUTE else 'false'}

## 4. Create the target schemas

Skipped for schemas that already exist.

In [ ]:
from databricks.sdk.errors import NotFound

for schema in (CDC_SCHEMA, QBC_SCHEMA):
    try:
        w.schemas.get(f'{CATALOG}.{schema}')
        print(f'{CATALOG}.{schema} exists')
    except NotFound:
        w.schemas.create(name=schema, catalog_name=CATALOG, comment='Created by the Tapworks minimal config example')
        print(f'{CATALOG}.{schema} created')

## 5. Generate the bundles

Same table list, two connectors. Without `cursor_columns`, the query-based connector reads each table as a full snapshot; add a `cursor_columns` column to `tables.csv` for incremental reads.

In [ ]:
cdc_df = run_pipeline_generation(
    connector_name='oracle_integrated',
    input_source=TABLES_CSV,
    output_dir=OUTPUT_DIR,
    targets=targets,
    default_values=cdc_defaults,
    override_config=overrides,
)

qbc_df = run_pipeline_generation(
    connector_name='oracle_query_based',
    input_source=TABLES_CSV,
    output_dir=OUTPUT_DIR,
    targets=targets,
    default_values=qbc_defaults,
    override_config=overrides,
)

print(cdc_df[['source_schema', 'source_table_name', 'target_table_name', 'pipeline_group', 'classic_compute']].to_string(index=False))

In [ ]:
print(open(f"{OUTPUT_DIR}/tapworks_oracle_cdc/resources/pipelines.yml").read())

## 6. Deploy and run

Each bundle is deployed with the Databricks CLI (`databricks bundle deploy`). In a Databricks workspace you can also open the generated bundle folder and deploy it from the workspace UI.

The `dev` target uses development mode: resource names get a `[dev <user>]` prefix and job schedules are paused, so the jobs only run when triggered below.

In [ ]:
def bundle(project, *args):
    cmd = ['databricks', 'bundle', *args, '-t', 'dev']
    if DATABRICKS_PROFILE:
        cmd += ['-p', DATABRICKS_PROFILE]
    result = subprocess.run(cmd, cwd=f'{OUTPUT_DIR}/{project}', capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout

job_ids = {}
for project in ('tapworks_oracle_cdc', 'tapworks_oracle_qbc'):
    bundle(project, 'deploy')
    summary = json.loads(bundle(project, 'summary', '-o', 'json'))
    for key, job in summary['resources']['jobs'].items():
        job_ids[job['name']] = job['id']
    print(f'Deployed {project}')

for name, job_id in job_ids.items():
    run = w.jobs.run_now(job_id=int(job_id))
    print(f'Started {name}: run {run.run_id}')

## 7. Check pipeline status

Integrated CDC updates can take up to about 30 minutes. Re-run this cell to refresh.

In [ ]:
for project in ('tapworks_oracle_cdc', 'tapworks_oracle_qbc'):
    summary = json.loads(bundle(project, 'summary', '-o', 'json'))
    for key, p in summary['resources']['pipelines'].items():
        info = w.pipelines.get(p['id'])
        latest = info.latest_updates[0].state.value if info.latest_updates else 'NOT STARTED'
        print(f"{info.name}: {latest}  {p.get('url', '')}")

## Clean up

```bash
databricks bundle destroy -t dev   # run in each folder under deployment/
```

Then drop the target schemas if they are no longer needed.